# Few-shot Prompts 

In [2]:
from langchain_core.prompts import PromptTemplate

example_prompt = PromptTemplate.from_template("질문 : {question} \n 답변 : {answer}")

In [7]:
examples = [
    {
        "question" : "주식 투자와 예금 중 어느 것이 더 수익률이 높은가?", 
        "answer" : """
후속 질문이 필요한가요: 네.
후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
후속 질문: 예금의 평균 이자율은 얼마인가요?
중간 답변: 예금의 평균 이자율은 연 1%입니다.
따라서 최종 답변은: 주식 투자
""",
    },
    {
        "question" : "부동산과 채권 중 어느 것이 더 안정적인 투자처인가?",
        "answer" : """
후속 질문이 필요한가요: 네.
후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
후속 질문: 채권의 위험도는 어느 정도인가요?
중간 답변: 채권의 위험도는 낮은 편입니다.
따라서 최종 답변은: 채권
        
        """
    }
]

In [10]:
print(example_prompt.invoke(examples[0]).to_string())

질문 : 주식 투자와 예금 중 어느 것이 더 수익률이 높은가? 
 답변 : 
후속 질문이 필요한가요: 네.
후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
후속 질문: 예금의 평균 이자율은 얼마인가요?
중간 답변: 예금의 평균 이자율은 연 1%입니다.
따라서 최종 답변은: 주식 투자



In [11]:
from langchain_core.prompts import FewShotPromptTemplate 

prompt = FewShotPromptTemplate(
    examples=examples, 
    example_prompt=example_prompt,
    suffix="질문 : {input}", 
    input_variables=['input'], 
)

print(prompt.invoke({'부동산 투자의 장점은 무엇인가?'}).to_string())

질문 : 주식 투자와 예금 중 어느 것이 더 수익률이 높은가? 
 답변 : 
후속 질문이 필요한가요: 네.
후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
후속 질문: 예금의 평균 이자율은 얼마인가요?
중간 답변: 예금의 평균 이자율은 연 1%입니다.
따라서 최종 답변은: 주식 투자


질문 : 부동산과 채권 중 어느 것이 더 안정적인 투자처인가? 
 답변 : 
후속 질문이 필요한가요: 네.
후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
후속 질문: 채권의 위험도는 어느 정도인가요?
중간 답변: 채권의 위험도는 낮은 편입니다.
따라서 최종 답변은: 채권

        

질문 : {'부동산 투자의 장점은 무엇인가?'}


In [ ]:
import numpy as np 
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

vector=embedding.embed_query('고양이')

arr = np.array(vector)
print(arr.shape)

(768,)


In [22]:

from langchain_chroma import Chroma 
from langchain_core.example_selectors import SemanticSimilarityExampleSelector 
from langchain_openai import OpenAIEmbeddings 

example_selector = SemanticSimilarityExampleSelector.from_examples(
    # 사용할 예제 목록 
    examples, 
    # 임베딩 생성에 사용하는 클래스 
    embedding,
    # 임베딩을 저장하고 유사도 검색을 수행하는 벡터 저장소
    Chroma, 
    # 선택할 예제의 수  
    k=1,    
)

question = "부동산 투자의 장점은 무엇인가?" 

selected_example = example_selector.select_examples({"question":question})

print(f"입력 질문 :  {question}")

for example in selected_example:
    print('\n')
    print('입력과 가장 유사하 예제')
    for k, v in reversed(example.items()):
        print(f"{k} : {v}")

입력 질문 :  부동산 투자의 장점은 무엇인가?


입력과 가장 유사하 예제
question : 부동산과 채권 중 어느 것이 더 안정적인 투자처인가?
answer : 
후속 질문이 필요한가요: 네.
후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
후속 질문: 채권의 위험도는 어느 정도인가요?
중간 답변: 채권의 위험도는 낮은 편입니다.
따라서 최종 답변은: 채권

        


In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate 

example_prompt = PromptTemplate(
    input_variables=["question", "answer"], 
    template="질문 : {question}\n답변 {answer}" 
)

prompt = FewShotPromptTemplate(
    example_selector = example_selector, 
    example_prompt = example_prompt, 
    prefix = "다음은 금융 관련 질문과 답변의 예입니다.",
    suffix = "질문: {input}\n 답변 :",
    input_variables = ["input"]
)

In [23]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gemma-3-4b-it",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

In [24]:
chain = prompt | llm

response =chain.invoke({"input" : "부동산 투자의 장점은 무엇인가요?"})

print(response.content)

답변을 드리겠습니다.

---
**질문 : 부동산 투자의 장점은 무엇인가요?**

**답변:** 부동산 투자는 여러 복합적인 이점을 제공하며, 투자 목표와 시장 상황에 따라 그 장점이 극대화될 수 있습니다. 주요 장점은 크게 세 가지로 볼 수 있습니다.

### 1. 현금 흐름 창출 (임대 수익 확보)
가장 즉각적이고 확실한 장점은 임대를 통한 **정기적인 현금 흐름(Cash Flow)**을 확보할 수 있다는 것입니다. 이는 투자를 통해 비교적 안정적인 생활비나 추가 수익을 꾸준히 창출하게 해줍니다.

### 2. 자본 가치 상승 (시세 차익)
부동산의 가장 근본적인 목표는 자산 가치 상승을 통한 시세 차익입니다. 우량 입지의 부동산은 시간이 지남에 따라 인플레이션과 함께 가치가 꾸준히 상승하는 경향이 있습니다. 이는 장기적인 부의 축적에 기여합니다.

### 3. 인플레이션 헤지 및 세제 혜택
*   **인플레이션 방어:** 물가 상승기에는 현금의 가치는 떨어지지만, 우량 부동산은 그 가치만큼이나 자산 가격도 상승하는 경향이 있어 인플레이션의 영향을 상쇄하고 부를 지키는 역할을 합니다.
*   **세제 혜택 (종류에 따라 다름):** 상황에 따라 취득세, 재산세 감면 또는 경비 처리 등을 통해 투자 과정에서 세금 측면의 이점을 누릴 수 있습니다.

***
**추가적으로 고려할 사항:**
물론, 부동산 투자는 이러한 장점만큼이나 높은 초기 투자 비용, 관리에 필요한 시간 및 비용(관리비, 공실 위험), 그리고 시장 리스크 등 고려해야 할 단점들도 있습니다. 투자를 결정하기 전에는 이러한 위험 요소들까지 균형 있게 분석하는 것이 중요합니다.
